In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

# level definition
# low, medium, high = 1, 2, 3

In [ ]:
# Prepare data

houses_by_lga = pd.read_csv("Houses by LGA.csv")
houses_by_lga.set_index(["LGA", "Year"], inplace=True)

offences_by_lga = pd.read_csv("Offences By LGA NEW.csv")
offences_by_lga.set_index(["LGA", "Year"], inplace=True)

egm_by_lga = pd.read_csv("EGM-New-Format.csv")
egm_by_lga.set_index(["LGA", "Year"], inplace=True)



merged_data = houses_by_lga.merge(offences_by_lga, how="inner", on=["LGA", "Year"]) \
                           .merge(egm_by_lga, how="inner", on=["LGA", "Year"])

merged_data.rename(columns={"total dollars" : "Total Loss On EGM", "rate" : "EGM Rate"}, inplace=True)

merged_data

In [ ]:
train = pd.concat([
    merged_data[merged_data.index.get_level_values("Year") == 2016],
    merged_data[merged_data.index.get_level_values("Year") == 2018],
    merged_data[merged_data.index.get_level_values("Year") == 2017],
   

])

test = pd.concat([
    
    merged_data[merged_data.index.get_level_values("Year") == 2019], 
    merged_data[merged_data.index.get_level_values("Year") == 2020],
])


low_weighted_house_price = train["Weighted House Price"].quantile(0.333)
high_weighted_house_price = train["Weighted House Price"].quantile(0.667)

def weighted_house_price_level_generator(price):
    if price < low_weighted_house_price:
        return "Low"
    elif low_weighted_house_price <= price < high_weighted_house_price:
        return "Medium"
    else:
        return "High"
    
merged_data["Weighted House Price Level"] = merged_data["Weighted House Price"].apply(weighted_house_price_level_generator)
merged_data

In [ ]:
train = pd.concat([
    merged_data[merged_data.index.get_level_values("Year") == 2016],
    merged_data[merged_data.index.get_level_values("Year") == 2018],
    merged_data[merged_data.index.get_level_values("Year") == 2017],
   

])

test = pd.concat([
    
    merged_data[merged_data.index.get_level_values("Year") == 2019], 
    merged_data[merged_data.index.get_level_values("Year") == 2020],
])

X_COLS = [
    "Offence Rate",
    "EGM Rate"
]

Y_COL = ["Weighted House Price Level"]

X_train = train[X_COLS]
y_train = train[Y_COL]

X_test = test[X_COLS]
y_test = test[Y_COL]

knn = KNeighborsClassifier(n_neighbors=3)

knn.fit(X_train, y_train)

# Calculate accuracy score for the test data
y_pred = knn.predict(X_test)
accuracy = knn.score(X_test, y_test)

accuracy
